# مشروع أسوان: نظام استرجاع وتوليد معزز بالمعرفة (RAG) عن ثقافة وسياحة أسوان
## من النصوص إلى إجابات موثوقة عن معابد ومعالم وعادات أسوان

هذا المشروع تطبيق عملي كامل لفكرة الـ RAG (Retrieval-Augmented Generation) بنفس منهجية
Lab 6 (الاسترجاع اللفظي) و Lab 7 (الاسترجاع الدلالي بالـ Embeddings) و Lab 8 (خط أنابيب RAG كامل)،
لكن مطبقة على قاعدة معرفة جديدة ومحايدة من تصميمنا: **أسوان — الثقافة، العادات والتقاليد،
المعابد، الحدائق، النزهات النيلية، والأنشطة السياحية**.

> السؤال الذي يجيب عنه هذا المشروع: كيف يمكن لنظام أن يفهم سؤال مستخدم عن أسوان
> (بالعامية أو الفصحى) ويسترجع أدق المعلومات من قاعدة معرفة نصية، ثم يبني منها إجابة؟


## الخطة العامة للمشروع (Pipeline)

```text
مستندات أسوان (نصوص)
→ تقطيع/تنظيف بسيط (Chunking)
→ تمثيل لفظي: TF-IDF و BM25
→ تمثيل دلالي: Sentence Embeddings
→ استرجاع هجين (Hybrid Retrieval)
→ تقييم بمقاييس Precision@K, Recall@K, Hit Rate@K, MRR
→ بناء حزمة سياق (Context Package)
→ بناء Prompt وإرساله لنموذج لغوي لتوليد إجابة نهائية (RAG)
→ نشر النظام كتطبيق ويب عبر Streamlit ورفعه على GitHub
```

### مصادر المعطيات
تم بناء قاعدة معرفة أصلية من **34 مستنداً** (وليست منسوخة من المعمل) تغطي:
معابد أسوان الأثرية، المسلة الناقصة، السد العالي وخزان أسوان، الحدائق والمتنزهات
(جزيرة كتشنر، كورنيش أسوان)، النزهات النيلية (الفلوكة)، المتاحف (المتحف النوبي، متحف أسوان)،
عادات وتقاليد النوبة (الضيافة، السبوع، الزفاف النوبي)، الحرف اليدوية، القرى والأسواق،
المطبخ المحلي، الطبيعة والمناخ، والمهرجانات.


## مخرجات التعلم المستهدفة

| الموضوع | ما يجب فهمه |
|---|---|
| Corpus | قاعدة المعرفة النصية (مستندات أسوان) |
| Query | سؤال المستخدم بالعربية |
| TF-IDF / BM25 | تمثيل واسترجاع لفظي (اعتماداً على تطابق الكلمات) |
| Sentence Embeddings | تمثيل دلالي متعدد اللغات يفهم المعنى وليس فقط الكلمات |
| Cosine Similarity | مقياس التشابه بين متجهي الاستعلام والمستند |
| Hybrid Retrieval | دمج النتائج اللفظية والدلالية بوزن alpha |
| Precision@K / Recall@K / Hit Rate@K / MRR | مقاييس تقييم جودة الاسترجاع |
| Context Package | حزمة النصوص المسترجعة الجاهزة لتُمرَّر لنموذج توليدي |
| RAG Prompt | صياغة تعليمات للنموذج التوليدي لبناء إجابة من السياق فقط |
| Streamlit Deployment | نشر النظام كتطبيق ويب تفاعلي مرتبط بـ GitHub |


## Section 1 — استيراد المكتبات

In [ ]:
# شغّل هذه الخلية أولاً على Google Colab (أو أي بيئة لا تحتوي على المكتبات مسبقاً)
!pip install -q rank_bm25 sentence-transformers scikit-learn pandas numpy


In [ ]:
# المكتبات اللفظية (TF-IDF / BM25)
import json
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

# المكتبة الدلالية (Sentence Embeddings متعددة اللغات لدعم العربية)
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 120)


> **ملاحظة تثبيت:** إذا لم تكن المكتبات مثبتة، شغّل الخلية التالية أولاً (تحتاج اتصال إنترنت،
> على سبيل المثال داخل Google Colab أو أي بيئة متصلة):
> ```
> !pip install -q sentence-transformers rank_bm25 scikit-learn pandas numpy
> ```


## Section 2 — بناء قاعدة المعرفة (Corpus)

In [ ]:
# هذه الخلية تنشئ ملفات قاعدة المعرفة تلقائياً داخل Colab (لا حاجة لرفع ملفات خارجية)
# قاعدة المعرفة محدثة الآن لتشمل مستندات مبنية على ورقتين علميتين محكّمتين عن التراث النوبي
import os, json
os.makedirs("data", exist_ok=True)

aswan_corpus = [{'document_id': 0, 'category': 'معابد', 'document': 'معبد فيلة يقع على جزيرة أجيليكيا بعد نقله من موقعه الأصلي إنقاذاً له من مياه بحيرة ناصر، وهو مخصص لعبادة الإلهة إيزيس ويعد من أجمل المعابد المصرية على الإطلاق.'}, {'document_id': 1, 'category': 'معابد', 'document': 'يقام في معبد فيلة عرض للصوت والضوء في المساء يحكي أسطورة إيزيس وأوزوريس، ويجذب هذا العرض آلاف الزوار سنوياً للاستمتاع بالإضاءة الملونة على أعمدة المعبد.'}, {'document_id': 2, 'category': 'معابد', 'document': 'معبدا أبو سمبل الكبير والصغير نحتهما رمسيس الثاني في الصخر تخليداً لانتصاراته وتكريماً لزوجته الملكة نفرتاري، ويقعان جنوب أسوان على بحيرة ناصر.'}, {'document_id': 3, 'category': 'معابد', 'document': 'تحدث ظاهرة تعامد الشمس على وجه رمسيس الثاني داخل معبد أبو سمبل مرتين كل عام، في الثاني والعشرين من فبراير والثاني والعشرين من أكتوبر، ويحضرها آلاف الزوار من أنحاء العالم.'}, {'document_id': 4, 'category': 'معابد', 'document': 'معبد كلابشة يعد من أكبر المعابد النوبية التي تم نقلها وإعادة بنائها بالقرب من السد العالي، وهو مخصص لعبادة الإله ماندوليس رب الشمس عند النوبيين.'}, {'document_id': 5, 'category': 'معابد', 'document': 'معبد بيت الوالي يقع بجوار معبد كلابشة وهو منحوت في الصخر، ونقشت جدرانه بمشاهد حربية تصور انتصارات رمسيس الثاني على النوبيين والليبيين.'}, {'document_id': 6, 'category': 'معالم أثرية', 'document': 'المسلة الناقصة تقع في المحاجر الشمالية بأسوان، وهي أكبر مسلة معروفة في مصر القديمة، تُظهر شقوقاً في الجرانيت أوقفت استكمال نحتها وفصلها عن الصخر.'}, {'document_id': 7, 'category': 'معالم أثرية', 'document': 'توضح المسلة الناقصة للزوار طريقة قدماء المصريين في استخراج المسلات من محاجر الجرانيت، وتعتبر شاهداً مهماً على تقنيات النحت والنقل في العصر الفرعوني.'}, {'document_id': 8, 'category': 'معالم أثرية', 'document': 'جزيرة الفنتين تقع وسط النيل أمام مدينة أسوان مباشرة، وتضم آثار معبد خنوم وقرى نوبية ملونة ومتحف أسوان الذي يعرض قطعاً من مختلف العصور.'}, {'document_id': 9, 'category': 'سدود', 'document': 'السد العالي بأسوان أحد أهم المشروعات الهندسية في القرن العشرين، شُيّد لحماية مصر من الفيضانات وتوليد الكهرباء وتخزين المياه في بحيرة ناصر.'}, {'document_id': 10, 'category': 'سدود', 'document': 'خزان أسوان القديم الذي بُني في بدايات القرن العشرين يقع جنوب مدينة أسوان مباشرة، ويعد من أوائل السدود الكبرى التي أقيمت على نهر النيل.'}, {'document_id': 11, 'category': 'حدائق ومتنزهات', 'document': 'جزيرة كتشنر، المعروفة أيضاً بالحديقة النباتية، تضم مئات الأنواع النادرة من النباتات والأشجار الاستوائية التي جمعها اللورد كتشنر من مختلف أنحاء العالم.'}, {'document_id': 12, 'category': 'حدائق ومتنزهات', 'document': 'يفضل الزوار التجول في الحديقة النباتية بجزيرة كتشنر عصراً حيث يمكن مشاهدة الطيور النادرة والاستمتاع بمناظر النيل الهادئة بين أشجار النخيل والمانجو.'}, {'document_id': 13, 'category': 'حدائق ومتنزهات', 'document': 'كورنيش أسوان يمتد على طول ضفة النيل الشرقية، وهو مكان مفضل للمشي والفسحة العائلية مساءً، ويضم مقاهي ومطاعم مطلة مباشرة على النهر.'}, {'document_id': 14, 'category': 'نزهات نيلية', 'document': 'رحلات الفلوكة في أسوان من أشهر أنشطة الترفيه، حيث يستقل السياح والزوار المحليون قوارب شراعية تقليدية للتجول بين الجزر الصخرية عند الغروب.'}, {'document_id': 15, 'category': 'نزهات نيلية', 'document': 'يمكن للزائر استئجار فلوكة من الكورنيش للإبحار حول جزيرة النباتات وجزيرة الفنتين، وتعد رحلة المساء وقت الغروب من أجمل التجارب في أسوان.'}, {'document_id': 16, 'category': 'متاحف', 'document': 'المتحف النوبي يعرض تاريخ وحضارة النوبة عبر العصور من خلال قطع أثرية ومجسمات وأزياء تقليدية، وقد أُنشئ ضمن مشروع إنقاذ آثار النوبة الدولي.'}, {'document_id': 17, 'category': 'متاحف', 'document': 'متحف أسوان يقع في مبنى قديم على جزيرة الفنتين ويعرض مقتنيات اكتُشفت في المنطقة، من تماثيل وأواني فخارية تعود لعصور فرعونية ورومانية مختلفة.'}, {'document_id': 18, 'category': 'ثقافة وعادات', 'document': 'يتميز النوبيون بألوانهم الزاهية في تزيين منازلهم، حيث تُطلى الجدران بألوان مبهجة وتُرسم عليها زخارف تمثل النخيل والتماسيح رمزاً للحماية والخصوبة.'}, {'document_id': 19, 'category': 'ثقافة وعادات', 'document': 'من عادات النوبيين الترحيب بالضيوف بتقديم القهوة أو الشاي فور الوصول، وتعد الضيافة جزءاً أساسياً من الهوية النوبية المعروفة بكرمها.'}, {'document_id': 20, 'category': 'ثقافة وعادات', 'document': 'يحتفظ سكان النوبة بلغتهم الخاصة وأغانيهم الشعبية المصحوبة بآلة الطار والدف، وتُنقل هذه التقاليد الشفهية من جيل إلى جيل حتى اليوم.'}, {'document_id': 21, 'category': 'ثقافة وعادات', 'document': 'السبوع هو احتفال تقليدي يقام في اليوم السابع لميلاد الطفل في أسوان والنوبة، حيث يجتمع الأهل والجيران وتُغنى أغاني خاصة بهذه المناسبة.'}, {'document_id': 22, 'category': 'ثقافة وعادات', 'document': 'تشتهر حفلات الزفاف النوبية بطقوس تمتد لعدة أيام، منها ليلة الحناء التي تُزيَّن فيها العروس بنقوش تقليدية وسط غناء ورقص جماعي.'}, {'document_id': 23, 'category': 'حرف يدوية', 'document': 'الحرف اليدوية النوبية مثل نسج السلال الملونة من سعف النخيل وصناعة الخرز والتطريز اليدوي تشكل مصدر دخل مهم للنساء في القرى النوبية.'}, {'document_id': 24, 'category': 'قرى وأسواق', 'document': 'القرى النوبية القريبة من أسوان مثل قرية غرب سهيل تجذب الزوار لمشاهدة البيوت الملونة والتفاعل مع التماسيح الأليفة التي يربيها بعض السكان.'}, {'document_id': 25, 'category': 'قرى وأسواق', 'document': 'سوق أسوان الشعبي يمتد بمحاذاة الكورنيش ويعرض التوابل الملونة والبخور والمنتجات النوبية اليدوية، ويعد وجهة مفضلة للتسوق والتجول مساءً.'}, {'document_id': 26, 'category': 'مطبخ محلي', 'document': 'الكركديه هو المشروب الأشهر في أسوان، يُصنع من نبات الكركديه المزروع محلياً، ويُقدَّم بارداً أو ساخناً وله شهرة تتجاوز حدود مصر.'}, {'document_id': 27, 'category': 'مطبخ محلي', 'document': 'يعتمد المطبخ النوبي على أطباق مثل الويكة والفتة النوبية والتمر واللحوم المطهوة ببطء، وتتميز الوجبات بنكهات مميزة موروثة عن أجداد النوبة.'}, {'document_id': 28, 'category': 'طبيعة', 'document': 'الجندل الأول هو منطقة الصخور الجرانيتية التي تعيق الملاحة جنوب أسوان، وتشكل مناظرها الطبيعية الفريدة مصدر جذب للزوار الباحثين عن هدوء النيل.'}, {'document_id': 29, 'category': 'طبيعة', 'document': 'تشتهر أسوان بمناخها الجاف الدافئ طوال العام تقريباً، مما يجعلها وجهة مثالية للسياحة الشتوية والاستشفاء من أمراض الروماتيزم والجهاز التنفسي.'}, {'document_id': 30, 'category': 'مهرجانات', 'document': 'مهرجان تعامد الشمس على معبد أبو سمبل يتحول إلى احتفال شعبي وفني كبير يضم عروضاً فلكلورية نوبية وفعاليات موسيقية تستمر طوال اليوم.'}, {'document_id': 31, 'category': 'أنشطة سياحية', 'document': 'تعد رحلة ركوب الجمال في الصحراء المحيطة بأسوان ومشاهدة غروب الشمس من أعلى التلال الرملية من الأنشطة المفضلة لدى السياح الباحثين عن مغامرة.'}, {'document_id': 32, 'category': 'أنشطة سياحية', 'document': 'بحيرة ناصر تتيح رحلات بحرية فاخرة لعدة أيام تمر بالمعابد النوبية المنقولة مثل أبو سمبل وكلابشة وعمدا، وتمنح تجربة مختلفة عن النيل التقليدي.'}, {'document_id': 33, 'category': 'مواصلات وموقع', 'document': 'تقع مدينة أسوان في أقصى جنوب مصر على الضفة الشرقية لنهر النيل، وتبعد عن القاهرة نحو تسعمائة كيلومتر ويمكن الوصول إليها بالطائرة أو القطار.'}, {'document_id': 34, 'category': 'عمارة نوبية', 'document': 'يُبنى البيت النوبي التقليدي من الطين اللبن على حافة الصحراء بين الجبل ونهر النيل، ويتكون غالباً من حوش مستطيل تحيط به الغرف، ومدخله الرئيسي يواجه النيل مباشرة.'}, {'document_id': 35, 'category': 'عمارة نوبية', 'document': 'تخلو أغلب البيوت النوبية من النوافذ الكبيرة، وتكتفي بفتحات علوية صغيرة تسمى الطاقات لتفادي برودة رياح الشتاء والحفاظ على برودة المنزل في فصل الصيف.'}, {'document_id': 36, 'category': 'زخارف ورموز', 'document': 'يزين النوبيون واجهات منازلهم برسوم هندسية ونباتية وحيوانية ملونة، ويُعتقد أن الغرض الأساسي منها هو درء الحسد وحماية أهل البيت من العين الحاسدة.'}, {'document_id': 37, 'category': 'زخارف ورموز', 'document': 'من أشهر الزخارف النوبية المثلث الذي يرمز للحماية من الحسد والسحر، والدائرة التي ترمز للشمس والقمر، والهلال الذي يوحي بالتفاؤل، والسمكة رمزاً للبعث وتجدد الحياة.'}, {'document_id': 38, 'category': 'حرف يدوية', 'document': 'تبرع النساء النوبيات في نسج أطباق وسلال من خوص النخيل ومراوح من الجريد، وتُعلَّق هذه المشغولات اليدوية على حوائط وأسقف المنزل لأغراض الزينة والحماية معاً.'}, {'document_id': 39, 'category': 'أزياء تقليدية', 'document': 'ترتدي المرأة النوبية جلباباً أسود واسع الأكمام يسمى التوب فوق جلباب آخر ملون، مع غطاء رأس يسمى الشبارة وخلخال في القدمين وحلية جلدية ملونة تتدلى على الصدر.'}, {'document_id': 40, 'category': 'أزياء تقليدية', 'document': 'يرتدي العريس النوبي يوم زفافه جلباباً أبيض وعمامة بيضاء ويحمل سيفاً وخنجراً رمزاً للشجاعة والرجولة، وينتعل حذاءً تقليدياً يسمى المركوب.'}, {'document_id': 41, 'category': 'طقوس الزواج', 'document': 'تبدأ مراسم الزواج النوبي بيوم يسمى بيرار نهار، حين تزور عائلة العريس بيت العروس حاملة الهدايا، ويُتفق خلاله على تفاصيل الزواج ومهر العروس وتُقرأ الفاتحة.'}, {'document_id': 42, 'category': 'طقوس الزواج', 'document': 'في مراسم تسمى أوكجر تطوف نساء عائلة العريس على بيوت القرية لإبلاغ الجيران بموعد الزفاف، حاملات طبقاً مزيناً بالزهور والعطور يسمى الكريدة.'}, {'document_id': 43, 'category': 'طقوس الزواج', 'document': 'ليلة الحناء عند النوبيين تسمى كوفري تور وتُعد من أقدس ليالي الزفاف، حيث تُزين العروس بالحناء وسط الغناء الفولكلوري والرقص الجماعي قبل ليلة الزفاف بيوم أو يومين.'}, {'document_id': 44, 'category': 'طقوس الزواج', 'document': 'من عادات الزواج النوبي أن يتحمل العريس الجزء الأكبر من تكاليف التجهيزات دون اشتراط قائمة عفش على العروس، وتندر حالات الطلاق في المجتمعات النوبية مقارنة بغيرها.'}, {'document_id': 45, 'category': 'طقوس الزواج', 'document': 'يفضّل المجتمع النوبي زواج الأقارب خصوصاً بين أبناء العمومة حفاظاً على تماسك العائلة والهوية النوبية، وهو ما يُعرف بنمط الزواج الداخلي.'}, {'document_id': 46, 'category': 'عادات المولود', 'document': 'بعد الولادة تمكث الأم فترة في بيت أهلها لترعاها والدتها، وتحرص العائلة على تقديم أطعمة مقوية خلال فترة النفاس وتجنّب خروج الأم من المنزل إلا للضرورة القصوى.'}, {'document_id': 47, 'category': 'طقوس الوفاة', 'document': 'عند وفاة أحد أفراد القرية النوبية يشارك جميع السكان في مراسم العزاء والدفن، ولا تحضر النساء عادة موكب تشييع الجنازة بل ينتظرن في المنزل لاستقبال المعزين.'}, {'document_id': 48, 'category': 'خلفية تاريخية', 'document': 'يُعتقد أن اسم النوبة مشتق من الكلمة المصرية القديمة نب وتعني الذهب، نسبة إلى مناجم الذهب التي اشتهرت بها المنطقة منذ العصور الفرعونية القديمة.'}, {'document_id': 49, 'category': 'خلفية تاريخية', 'document': 'ينقسم سكان النوبة المصرية تقليدياً إلى ثلاث مجموعات رئيسية هي الكنوز والفديجا والعرب، ولكل مجموعة لهجتها وبعض عاداتها الخاصة رغم تجاورها الجغرافي على امتداد النيل.'}, {'document_id': 50, 'category': 'طب شعبي', 'document': 'اعتمد أهل النوبة تاريخياً على طب شعبي مستمد من الأعشاب والنباتات البرية المحلية لعلاج بعض الأعراض البسيطة، إلى جانب استفادتهم من الخدمات الصحية الحديثة المتوفرة حالياً.'}]

aswan_queries = [{'query': 'معبد فيلة وإيزيس', 'relevant_ids': [0, 1], 'type': 'كلمات مفتاحية'}, {'query': 'ما هي ظاهرة تعامد الشمس في أبو سمبل؟', 'relevant_ids': [2, 3, 30], 'type': 'سؤال طبيعي'}, {'query': 'أين يمكنني التنزه بجانب النيل مساءً؟', 'relevant_ids': [13, 14, 15], 'type': 'عدم تطابق مفردات'}, {'query': 'الحديقة النباتية جزيرة كتشنر', 'relevant_ids': [11, 12], 'type': 'كلمات مفتاحية'}, {'query': 'عادات وتقاليد أهل النوبة في الاحتفالات', 'relevant_ids': [18, 19, 20, 21, 22], 'type': 'سؤال طبيعي'}, {'query': 'احتفال السبوع عند ولادة الطفل', 'relevant_ids': [21], 'type': 'كلمات مفتاحية'}, {'query': 'أماكن بيع الهدايا والمنتجات اليدوية في أسوان', 'relevant_ids': [23, 25], 'type': 'عدم تطابق مفردات'}, {'query': 'ما فائدة السد العالي؟', 'relevant_ids': [9], 'type': 'سؤال طبيعي'}, {'query': 'المتحف النوبي وتاريخ النوبة', 'relevant_ids': [16], 'type': 'كلمات مفتاحية'}, {'query': 'رحلة بحرية على بحيرة ناصر لعدة أيام', 'relevant_ids': [32], 'type': 'سؤال طبيعي'}, {'query': 'مشروب الكركديه ومطبخ أسوان', 'relevant_ids': [26, 27], 'type': 'كلمات مفتاحية'}, {'query': 'نشاط مغامرة في الصحراء وقت الغروب', 'relevant_ids': [31], 'type': 'عدم تطابق مفردات'}, {'query': 'كيف يستقبل النوبيون ضيوفهم؟', 'relevant_ids': [19], 'type': 'سؤال طبيعي'}, {'query': 'أكبر مسلة لم تكتمل في مصر القديمة', 'relevant_ids': [6, 7], 'type': 'سؤال طبيعي'}, {'query': 'شكل البيت النوبي وطريقة بنائه', 'relevant_ids': [34, 35], 'type': 'سؤال طبيعي'}, {'query': 'لماذا تخلو البيوت النوبية من النوافذ الكبيرة', 'relevant_ids': [35], 'type': 'عدم تطابق مفردات'}, {'query': 'معاني الزخارف والرموز في الفن النوبي', 'relevant_ids': [36, 37], 'type': 'سؤال طبيعي'}, {'query': 'ملابس المرأة والرجل النوبي التقليدية', 'relevant_ids': [39, 40], 'type': 'كلمات مفتاحية'}, {'query': 'مراسم ليلة الحنة عند النوبيين', 'relevant_ids': [43], 'type': 'كلمات مفتاحية'}, {'query': 'خطوات الزواج النوبي بالترتيب من الخطبة للزفاف', 'relevant_ids': [41, 42, 43, 44], 'type': 'سؤال طبيعي'}, {'query': 'أصل تسمية النوبة ومعناها', 'relevant_ids': [48], 'type': 'سؤال طبيعي'}, {'query': 'القبائل والمجموعات السكانية في النوبة', 'relevant_ids': [49], 'type': 'كلمات مفتاحية'}, {'query': 'هل يفضل النوبيون زواج الأقارب؟', 'relevant_ids': [45], 'type': 'سؤال طبيعي'}, {'query': 'عادات استقبال المولود الجديد عند النوبيين', 'relevant_ids': [46], 'type': 'سؤال طبيعي'}, {'query': 'طقوس الدفن ومشاركة القرية في العزاء', 'relevant_ids': [47], 'type': 'سؤال طبيعي'}]

with open("data/aswan_corpus.json", "w", encoding="utf-8") as f:
    json.dump(aswan_corpus, f, ensure_ascii=False, indent=1)

with open("data/aswan_queries.json", "w", encoding="utf-8") as f:
    json.dump(aswan_queries, f, ensure_ascii=False, indent=1)

print(f"تم إنشاء {len(aswan_corpus)} مستند و {len(aswan_queries)} استعلام داخل مجلد data/")


In [ ]:
with open("data/aswan_corpus.json", "r", encoding="utf-8") as f:
    corpus_raw = json.load(f)

documents_df = pd.DataFrame(corpus_raw)
documents = documents_df["document"].tolist()

print("عدد المستندات:", len(documents))
documents_df.head(10)


## Section 3 — الاستعلامات (Queries) والحقيقة المرجعية (Ground Truth)

In [ ]:
with open("data/aswan_queries.json", "r", encoding="utf-8") as f:
    queries_raw = json.load(f)

queries_df = pd.DataFrame(queries_raw)
print("عدد الاستعلامات:", len(queries_df))
queries_df


تلاحظ أن بعض الاستعلامات مصممة عمداً بمفردات مختلفة عن المستند (مثال: سؤال
"أين يمكنني التنزه بجانب النيل مساءً؟" لا يستخدم كلمة "فلوكة" أو "كورنيش" حرفياً)،
وهذا لاختبار أين يفشل الاسترجاع اللفظي وأين ينجح الاسترجاع الدلالي — بالضبط كما في Lab 7.

## Section 4 — مقاييس التقييم (Retrieval Metrics)

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    hits = len(set(top_k) & set(relevant_ids))
    return hits / k

def recall_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    hits = len(set(top_k) & set(relevant_ids))
    return hits / len(relevant_ids)

def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    return 1.0 if len(set(top_k) & set(relevant_ids)) > 0 else 0.0

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


## Section 4.5 — تجهيز النصوص (Arabic Text Preprocessing)

قبل بناء أي تمثيل لفظي (TF-IDF أو BM25) لازم نوحّد شكل الكلمات، لأن بدون هذا يعامل النظام
"النيل" و"النيلية" و"نيل" كأنهم ثلاث كلمات مختلفة تماماً رغم أنهم من نفس الجذر. خطوات التنظيف:

- إزالة التشكيل (الحركات).
- توحيد أشكال الألف (إ، أ، آ ← ا).
- توحيد الألف المقصورة بالياء (ى ← ي).
- توحيد التاء المربوطة بالهاء (ة ← ه) — تسهّل المطابقة رغم أنها تفقد بعض الدقة الإملائية.
- إزالة علامات الترقيم والمسافات الزائدة.

**ملاحظة:** الاسترجاع الدلالي (Embeddings) في Section 7 يستخدم النص الأصلي بدون هذا التنظيف،
لأن نموذج الـ Sentence Transformer قوي بما يكفي ليتعامل مع النص الخام، والتنظيف الزائد
أحياناً يفقده بعض الإشارات السياقية المفيدة.

In [ ]:
import re

def preprocess_arabic(text):
    text = re.sub(r'[\u064B-\u065F]', '', text)      # إزالة التشكيل (الحركات)
    text = re.sub(r'[إأآا]', 'ا', text)                 # توحيد أشكال الألف
    text = re.sub(r'ى', 'ي', text)                      # توحيد الألف المقصورة بالياء
    text = re.sub(r'ة', 'ه', text)                      # توحيد التاء المربوطة بالهاء
    text = re.sub(r'[^\w\s]', '', text)                # إزالة علامات الترقيم
    text = re.sub(r'\s+', ' ', text).strip()             # إزالة المسافات الزائدة
    return text

documents_clean = [preprocess_arabic(doc) for doc in documents]

print("قبل التنظيف:", documents[0])
print("بعد التنظيف :", documents_clean[0])

## Section 5 — الاسترجاع اللفظي رقم 1: TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents_clean)

def tfidf_retrieve(query, k=3):
    query_vec = tfidf_vectorizer.transform([preprocess_arabic(query)])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    ranked_ids = np.argsort(scores)[::-1]
    return list(ranked_ids[:k]), scores

print("شكل مصفوفة TF-IDF:", tfidf_matrix.shape)


## Section 6 — الاسترجاع اللفظي رقم 2: BM25

In [ ]:
tokenized_corpus = [doc.split() for doc in documents_clean]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query, k=3):
    tokenized_query = preprocess_arabic(query).split()
    scores = bm25.get_scores(tokenized_query)
    ranked_ids = np.argsort(scores)[::-1]
    return list(ranked_ids[:k]), scores


## Section 7 — الاسترجاع الدلالي: Sentence Embeddings

نستخدم نموذج `paraphrase-multilingual-MiniLM-L12-v2` لأنه يدعم اللغة العربية بشكل جيد
ويحوّل كل جملة إلى متجه كثيف (Dense Vector) يمثل المعنى العام للجملة، وليس فقط الكلمات
الموجودة فيها — هذا هو الفرق الجوهري عن TF-IDF و BM25.

In [ ]:
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

document_embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
print("شكل متجهات المستندات:", document_embeddings.shape)

def embedding_retrieve(query, k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, document_embeddings)[0]
    ranked_ids = np.argsort(scores)[::-1]
    return list(ranked_ids[:k]), scores


## Section 8 — الاسترجاع الهجين (Hybrid Retrieval)

ندمج درجات BM25 (بعد تطبيعها بين 0 و1) مع درجات التشابه الدلالي بوزن `alpha`:

`hybrid_score = alpha * semantic_score + (1 - alpha) * lexical_score`


In [ ]:
def normalize(scores):
    scores = np.array(scores, dtype=float)
    if scores.max() - scores.min() == 0:
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())

def hybrid_retrieve(query, k=3, alpha=0.6):
    _, bm25_scores = bm25_retrieve(query, k=len(documents))
    _, semantic_scores = embedding_retrieve(query, k=len(documents))

    lexical_norm = normalize(bm25_scores)
    semantic_norm = normalize(semantic_scores)

    hybrid_scores = alpha * semantic_norm + (1 - alpha) * lexical_norm
    ranked_ids = np.argsort(hybrid_scores)[::-1]
    return list(ranked_ids[:k]), hybrid_scores


## Section 9 — تقييم المسترجعات الثلاثة على كل الاستعلامات

In [ ]:
def evaluate_retriever(retrieve_fn, k=3, **kwargs):
    precisions, recalls, hits, rrs = [], [], [], []
    for _, row in queries_df.iterrows():
        retrieved_ids, _ = retrieve_fn(row["query"], k=k, **kwargs)
        relevant_ids = row["relevant_ids"]
        precisions.append(precision_at_k(retrieved_ids, relevant_ids, k))
        recalls.append(recall_at_k(retrieved_ids, relevant_ids, k))
        hits.append(hit_rate_at_k(retrieved_ids, relevant_ids, k))
        rrs.append(reciprocal_rank(retrieved_ids, relevant_ids))
    return {
        "Precision@K": np.mean(precisions),
        "Recall@K": np.mean(recalls),
        "HitRate@K": np.mean(hits),
        "MRR": np.mean(rrs),
    }

K = 3
results = pd.DataFrame({
    "TF-IDF": evaluate_retriever(tfidf_retrieve, k=K),
    "BM25": evaluate_retriever(bm25_retrieve, k=K),
    "Embeddings": evaluate_retriever(embedding_retrieve, k=K),
    "Hybrid (alpha=0.6)": evaluate_retriever(hybrid_retrieve, k=K, alpha=0.6),
}).T

results


### القراءة المتوقعة للنتائج
- استعلامات الكلمات المفتاحية المباشرة (مثال: "المتحف النوبي وتاريخ النوبة") يُتوقع أن ينجح
  فيها TF-IDF و BM25 بسهولة.
- استعلامات **عدم تطابق المفردات** (مثال: "أين يمكنني التنزه بجانب النيل مساءً؟") هي التي
  يُتوقع أن يتفوق فيها الاسترجاع الدلالي (Embeddings) بوضوح، لأنه يفهم أن "التنزه بجانب النيل"
  قريب دلالياً من "الفلوكة" و"الكورنيش" حتى بدون تطابق كلمات.
- الاسترجاع الهجين متوقع أن يحقق أفضل أداء عام لأنه يستفيد من مزايا الطريقتين معاً.

## Section 10 — مقارنة أفضل نتيجة (Top-1) على استعلام عدم تطابق مفردات

In [ ]:
sample_query = "أين يمكنني التنزه بجانب النيل مساءً؟"

for name, fn, kwargs in [
    ("TF-IDF", tfidf_retrieve, {}),
    ("BM25", bm25_retrieve, {}),
    ("Embeddings", embedding_retrieve, {}),
    ("Hybrid", hybrid_retrieve, {"alpha": 0.6}),
]:
    ids, _ = fn(sample_query, k=1, **kwargs)
    print(f"{name:12s} -> {documents[ids[0]]}")


## Section 11 — بناء حزمة السياق (Context Package) لكل استعلام

حزمة السياق هي النصوص المسترجعة مرتّبة مع بيانات وصفية (رقم المستند، الفئة)، وهي ما
سيُمرَّر فعلياً للنموذج التوليدي بدل تمرير قاعدة المعرفة كلها.

In [ ]:
def build_context_package(query, k=3, alpha=0.6):
    ids, scores = hybrid_retrieve(query, k=k, alpha=alpha)
    context_chunks = []
    for rank, doc_id in enumerate(ids, start=1):
        context_chunks.append({
            "rank": rank,
            "document_id": int(doc_id),
            "category": documents_df.loc[doc_id, "category"],
            "text": documents[doc_id],
            "score": float(scores[doc_id]),
        })
    return context_chunks

context_package = build_context_package("عادات وتقاليد أهل النوبة في الاحتفالات", k=3)
pd.DataFrame(context_package)


## Section 12 — صياغة الـ Prompt لنموذج لغوي (RAG)

In [ ]:
def build_rag_prompt(query, context_package):
    context_text = "\n".join(
        f"[{c['rank']}] ({c['category']}) {c['text']}" for c in context_package
    )
    prompt = f"""أنت مرشد سياحي وثقافي متخصص في مدينة أسوان.
أجب عن سؤال المستخدم بالاعتماد فقط على المعلومات الموجودة في السياق أدناه.
إذا لم تكفِ المعلومات للإجابة، صرّح بذلك بوضوح ولا تختلق معلومات من عندك.

السياق:
{context_text}

سؤال المستخدم: {query}

الإجابة:"""
    return prompt

print(build_rag_prompt("عادات وتقاليد أهل النوبة في الاحتفالات", context_package))


## Section 13 — استدعاء نموذج Claude لتوليد الإجابة النهائية (RAG كامل)

هذه الخلية اختيارية وتحتاج مفتاح API من Anthropic. اضبط متغير البيئة `ANTHROPIC_API_KEY`
قبل التشغيل، أو مرّر المفتاح مباشرة. هذه نفس الفكرة المطبّقة داخل تطبيق Streamlit
(`app.py`) المرفق مع المشروع.

In [ ]:
import os
# from anthropic import Anthropic
# client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

def generate_rag_answer(query, k=3, alpha=0.6):
    context_package = build_context_package(query, k=k, alpha=alpha)
    prompt = build_rag_prompt(query, context_package)

    # مثال حقيقي لاستدعاء Claude (فعّله بعد ضبط مفتاح الـ API):
    # response = client.messages.create(
    #     model="claude-sonnet-4-6",
    #     max_tokens=500,
    #     messages=[{"role": "user", "content": prompt}],
    # )
    # answer = response.content[0].text

    answer = "[هنا تظهر إجابة النموذج التوليدي بعد تفعيل مفتاح الـ API]"
    return answer, context_package

answer, used_context = generate_rag_answer("ما هي ظاهرة تعامد الشمس في أبو سمبل؟")
print(answer)


## Section 14 — تحليل حالات الفشل (Error Analysis)

اختر ثلاثة استعلامات ضعيفة الأداء بعد تشغيل التقييم أعلاه، وحلّل لكل منها:

1. ما هو المستند الصحيح المتوقع؟
2. ما الذي استرجعه كل نظام فعلياً؟
3. لماذا فشل النظام اللفظي أو الدلالي في هذه الحالة بالتحديد (تطابق كلمات جزئي، غياب سياق، تشابه فئات متقاربة، إلخ)؟

مثال جاهز للتحليل: الاستعلام "نشاط مغامرة في الصحراء وقت الغروب" لا يحتوي على كلمة
"جمال" الموجودة في المستند رقم 31 — راقب كيف يتعامل كل نظام مع هذا الغياب.

## الخلاصات النهائية

1. الاسترجاع اللفظي (TF-IDF, BM25) قوي في استعلامات الكلمات المفتاحية المباشرة لكنه يفشل عند تغيّر المفردات.
2. الاسترجاع الدلالي (Embeddings) يفهم المعنى ويتفوق في الأسئلة الطبيعية وحالات عدم تطابق المفردات.
3. الاسترجاع الهجين غالباً يحقق أفضل توازن بين الدقة والتغطية.
4. جودة إجابة نظام RAG محكومة بجودة الاسترجاع أولاً — سياق ضعيف يعني إجابة ضعيفة مهما كان النموذج التوليدي قوياً.
5. صياغة الـ Prompt بوضوح (تقييد الإجابة بالسياق المسترجع فقط) تقلل من اختلاق النموذج لمعلومات غير موجودة.
6. الخطوة التالية الطبيعية بعد بناء وتقييم النظام محلياً هي نشره كتطبيق تفاعلي يصل إليه أي مستخدم عبر الإنترنت — وهذا ما يقوم به `app.py` عبر Streamlit، مع رفع المشروع كاملاً على GitHub.


## المراجع
- scikit-learn `TfidfVectorizer`: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- scikit-learn `cosine_similarity`: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- rank_bm25: https://github.com/dorianbrown/rank_bm25
- Sentence-Transformers: https://www.sbert.net/
- Streamlit: https://docs.streamlit.io/
- Anthropic API: https://docs.claude.com/
